In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

### Installation

In [1]:
%%capture
!pip install unsloth
# Also get the latest nightly Unsloth!
!pip install --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

### Unsloth

In [2]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 3072 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

model, tokenizer = FastLanguageModel.from_pretrained(
    # model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
    model_name = "unsloth/DeepSeek-R1-Distill-Llama-8B-unsloth-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.1.8: Fast Llama patching. Transformers: 4.48.2.
   \\   /|    GPU: Tesla T4. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu121. CUDA: 7.5. CUDA Toolkit: 12.1. Triton: 3.1.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post1. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.96G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/236 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.9k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

### Dataset Prep

In [5]:
import json

paths = {
    "gsm8k": "/kaggle/input/llmbenchmarks/gsm8k_sample.jsonl",
    "odyssey": "/kaggle/input/llmbenchmarks/odyssey_sample.jsonl",
    "mbpp": "/kaggle/input/llmbenchmarks/mbpp_sample.jsonl",
    "human-eval":"/kaggle/input/llmbenchmarks/humaneval_sample.jsonl",
    "big-navigate":"/kaggle/input/llmbenchmarks/big-navigate_sample.jsonl",
    "big-reasoning-color":"/kaggle/input/llmbenchmarks/big-reasoning-color_sample.jsonl"
}

curr_benchmark = "human-eval"

path = paths[curr_benchmark]
dataset = []
with open(path, 'r') as infile:
    for line in infile:
        dataset.append(json.loads(line))
        dataset.append(json.loads(line))
        dataset.append(json.loads(line))
dataset = dataset[:300]
print(len(dataset))

300


In [6]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [7]:
max_new_length = 2600
slicing = 20

alpaca_prompt = """Give a response of code according to the instruction.

### Instruction:
{}

### Response:
{}"""

In [3]:
# test_prompt = """Given a set of movement instructions, output a response of a boolean whether or not the subject returns to the original position or not.

# ### Instruction:
# \ndef max_fill(grid, capacity):\n    import math\n    \"\"\"\n    You are given a rectangular grid of wells. Each row represents a single well,\n    and each 1 in a row represents a single unit of water.\n    Each well has a corresponding bucket that can be used to extract water from it, \n    and all buckets have the same capacity.\n    Your task is to use the buckets to empty the wells.\n    Output the number of times you need to lower the buckets.\n\n    Example 1:\n        Input: \n            grid : [[0,0,1,0], [0,1,0,0], [1,1,1,1]]\n            bucket_capacity : 1\n        Output: 6\n\n    Example 2:\n        Input: \n            grid : [[0,0,1,1], [0,0,0,0], [1,1,1,1], [0,1,1,1]]\n            bucket_capacity : 2\n        Output: 5\n    \n    Example 3:\n        Input: \n            grid : [[0,0,0], [0,0,0]]\n            bucket_capacity : 5\n        Output: 0\n\n    Constraints:\n        * all wells have the same length\n        * 1 <= grid.length <= 10^2\n        * 1 <= grid[:,1].length <= 10^2\n        * grid[i][j] -> 0 | 1\n        * 1 <= capacity <= 10\n    \"\"\"\n

# ### Response:
# """

# # Function to count tokens
# def count_tokens(text):
#     tokens = tokenizer(text, return_tensors="pt")["input_ids"]
#     num_tokens = tokens.shape[1]  # Token count
#     return num_tokens

# # Example input
# token_count = count_tokens(test_prompt)
# print(f"Number of tokens: {token_count}")

Number of tokens: 351


In [8]:
FastLanguageModel.for_inference(model)

item = dataset[0]

formatted_prompts = [
    alpaca_prompt.format(item["prompt"], item["placeholder"])
]

inputs = tokenizer(
    formatted_prompts,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=max_seq_length - max_new_length
).to(device)

outputs = model.generate(
    **inputs, 
    max_new_tokens = max_new_length, 
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id
)

decoded_response = tokenizer.batch_decode(outputs, skip_special_tokens=True)

print(decoded_response)

['Give a response of code according to the instruction.\n\n### Instruction:\n\ndef eat(number, need, remaining):\n    """\n    You\'re a hungry rabbit, and you already have eaten a certain number of carrots,\n    but now you need to eat more carrots to complete the day\'s meals.\n    you should return an array of [ total number of eaten carrots after your meals,\n                                    the number of carrots left after your meals ]\n    if there are not enough remaining carrots, you will eat all remaining carrots, but will still be hungry.\n    \n    Example:\n    * eat(5, 6, 10) -> [11, 4]\n    * eat(4, 8, 9) -> [12, 1]\n    * eat(1, 10, 10) -> [11, 0]\n    * eat(2, 11, 5) -> [7, 0]\n    \n    Variables:\n    @number : integer\n        the number of carrots that you have eaten.\n    @need : integer\n        the number of carrots that you need to eat.\n    @remaining : integer\n        the number of remaining carrots thet exist in stock\n    \n    Constrain:\n    * 0 <= num

In [9]:
torch.cuda.empty_cache()

In [ ]:
FastLanguageModel.for_inference(model)
decoded_responses = []
# for data in dataset:
i = 0
for _ in range(len(dataset)//slicing):
    data = dataset[i:i+slicing]
    i += slicing
    formatted_prompts = [alpaca_prompt.format(item['prompt'], item['placeholder']) for item in data]

    inputs = tokenizer(
        formatted_prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_seq_length - max_new_length
    ).to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_length,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id
    )

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    for _, response in enumerate(decoded):
        decoded_responses.append(response)

    del inputs, outputs
    torch.cuda.empty_cache()
    print(f"processed line {i-slicing} - {i}")

In [21]:
import json

jsonl_data = [
    {
        "task_id": item["task_id"],
        "completion": response.split("### Response:")[-1].strip(),
    }
    for item, response in zip(dataset, decoded_responses)
]

with open(f"deepseek_{curr_benchmark}_responses.jsonl", "w") as f:
    for entry in jsonl_data:
        f.write(json.dumps(entry) + "\n")  # Write one JSON object per line

# Graveyard

In [ ]:
# from unsloth import FastLanguageModel
# import torch
# max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
# dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
# load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
#     max_seq_length = max_seq_length,
#     dtype = dtype,
#     load_in_4bit = load_in_4bit,
#     # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
# )

# alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

# ### Instruction:
# {}

# ### Response:
# {}"""

# inputs = tokenizer([
#     alpaca_prompt.format(
#             data["instruction"],
#             data["placeholder"],
#         )
# ], return_tensors = "pt").to("cuda")

# outputs = model.generate(**inputs, max_new_tokens = 64, use_cache = True)
# tokenizer.batch_decode(outputs)


# inputs = tokenizer(alpaca_data, return_tensors = "pt").to("cuda")

# inputs = tokenizer(
# [
#     alpaca_prompt.format(
#         "Continue the fibonnaci sequence.", # instruction
#         "1, 1, 2, 3, 5, 8", # input
#         "", # output - leave this blank for generation!
#     )
# ], return_tensors = "pt").to("cuda")

In [ ]:
# formatted_prompts = [
#     alpaca_prompt.format(item["instruction"], item["output"]) for item in dataset
# ]

# inputs = tokenizer(
#     formatted_prompts,
#     return_tensors="pt",
#     padding=True,
#     truncation=True,
# ).to(device)

# outputs = model.generate(**inputs, max_new_tokens = 200, use_cache = True)
# decoded_responses = tokenizer.batch_decode(outputs, skip_special_tokens=True)

# for i, response in enumerate(decoded_responses):
#     print(f"Response {i+1}:\n{response}\n{'-'*100}")